In [ ]:

import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
from cuml.cluster import HDBSCAN

from selectzyme.backend.embed import gen_embedding

In [ ]:
def import_results(dataset_name: str) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    """
    Imports and loads results from a dataset downloaded from Hugging Face Hub.
    Args:
        dataset_name (str): Name of the dataset to fetch from Hugging Face Hub.
    Returns:
        tuple: A tuple containing the following:
            - pd.DataFrame: DataFrame loaded from "df.parquet".
            - np.ndarray: Reduced feature matrix loaded from "x_red_mst_slc.npz".
            - np.ndarray: Minimum spanning tree (MST) loaded from "x_red_mst_slc.npz".
            - np.ndarray: Linkage matrix loaded from "x_red_mst_slc.npz".
    """
    # Download files from Hugging Face Hub
    df_path = hf_hub_download(repo_id="davari-group/selectzyme-app-data", 
                              filename=f"{dataset_name}/df.parquet", 
                              repo_type="dataset")
    npz_path = hf_hub_download(repo_id="davari-group/selectzyme-app-data", 
                               filename=f"{dataset_name}/x_red_mst_slc.npz", 
                               repo_type="dataset")
    
    # Load data
    df = pd.read_parquet(df_path)
    adata = np.load(npz_path)
    X_red = adata["X_red"]
    mst = adata["mst"]
    linkage = adata["linkage"]

    return df, X_red, mst, linkage

In [ ]:
dataset_name = "lov"
plm = "prott5"
df, X_red, mst, linkage = import_results(dataset_name)
#df = df.head(400)
X = gen_embedding(df["sequence"].tolist(), plm_model=plm)

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import pandas as pd

def knn_label_consistency_scores(X, labels, k=8, n_perm=200, seed=0):
    """
    Evaluate KNN label consistency in feature space.
    
    Args:
        X: Feature matrix (n_samples, n_features)
        labels: Cluster labels array
        k: Number of neighbors
        n_perm: Number of permutations for null distribution
        seed: Random seed
        
    Returns:
        real: Real KNN label agreement score
        null_mean: Mean of permutation null distribution
        std: Standard deviation of null distribution
    """
    rng = np.random.default_rng(seed)
    
    # Handle missing labels
    valid_idx = labels >= 0  # Exclude noise points (-1 label in HDBSCAN)
    X_valid = X[valid_idx]
    y = labels[valid_idx]
    
    if len(np.unique(y)) < 2:
        return np.nan, np.nan, np.nan
    
    # Build KNN in feature space
    knn = NearestNeighbors(n_neighbors=k+1).fit(X_valid)  # k+1 because first neighbor is itself
    neigh_idx = knn.kneighbors(return_distance=False)[:, 1:]  # Skip self
    
    # Real score: agreement between neighbors
    real = np.mean([(y[neigh] == y[i]).mean() for i, neigh in enumerate(neigh_idx)])
    
    # Null distribution: random permutation of labels
    null_scores = []
    for _ in range(n_perm):
        y_perm = rng.permutation(y)
        s = np.mean([(y_perm[neigh] == y_perm[i]).mean() for i, neigh in enumerate(neigh_idx)])
        null_scores.append(s)
    
    null_mean = float(np.mean(null_scores))
    std = float(np.std(null_scores, ddof=1)) if len(null_scores) > 1 else np.nan
    
    return real, null_mean, std


def compute_label_correlations(labels_dict):
    """
    Compute label agreement metrics between different clusterings.
    
    Args:
        labels_dict: Dictionary of {clustering_name: labels_array}
        
    Returns:
        DataFrame with ARI and NMI correlations
    """
    results = []
    names = list(labels_dict.keys())
    
    for i, name1 in enumerate(names):
        for name2 in names[i+1:]:
            labels1 = labels_dict[name1]
            labels2 = labels_dict[name2]
            
            # Only compare valid clusters (exclude -1 noise label)
            valid_idx = (labels1 >= 0) & (labels2 >= 0)
            
            if valid_idx.sum() > 0:
                ari = adjusted_rand_score(labels1[valid_idx], labels2[valid_idx])
                nmi = normalized_mutual_info_score(labels1[valid_idx], labels2[valid_idx])
            else:
                ari, nmi = np.nan, np.nan
            
            results.append({
                'clustering1': name1,
                'clustering2': name2,
                'ARI': ari,
                'NMI': nmi
            })
    
    return pd.DataFrame(results)


In [ ]:
# Evaluate KNN agreement stability over clustering parameters
min_cluster_size = list(range(2, 17, 2))
min_samples = list(range(1, 16, 2))

knn_grid_results = []
labels_by_params = {}

for min_size in min_cluster_size:
    for min_samp in min_samples:
        clusterer = HDBSCAN(
            min_samples=min_samp,
            min_cluster_size=min_size,
            cluster_selection_method="leaf",
            gen_min_span_tree=True,
        )
        clusterer.fit(X)
        labels = clusterer.labels_
        labels_by_params[(min_size, min_samp)] = labels
        
        # Track noise points (label == -1)
        total_points = len(labels)
        noise_count = int((labels == -1).sum())
        noise_fraction = float(noise_count) / float(total_points) if total_points > 0 else np.nan
        
        real, null_mean, std = knn_label_consistency_scores(X, labels, k=10, n_perm=200)
        knn_grid_results.append({
            "min_cluster_size": min_size,
            "min_samples": min_samp,
            "knn_agreement": real,
            "permutation_null_mean": null_mean,
            "permutation_null_std": std,
            "delta_to_null": real - null_mean if pd.notna(real) and pd.notna(null_mean) else np.nan,
            "noise_count": noise_count,
            "noise_fraction": noise_fraction,
        })
        
        print(
            f"min_cluster_size={min_size}, min_samples={min_samp}: "
            f"KNN agreement={real:.4f}, permutation null={null_mean:.4f} ± {std:.4f}, "
            f"noise={noise_count}/{total_points} ({noise_fraction:.3f})"
        )

knn_grid_df = pd.DataFrame(knn_grid_results)
knn_grid_df = knn_grid_df.sort_values(["min_samples", "min_cluster_size"]).reset_index(drop=True)
print("\nKNN agreement grid:")
print(knn_grid_df)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pivot results into matrices for plotting
real_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="knn_agreement").reindex(index=min_samples, columns=min_cluster_size)
delta_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="delta_to_null").reindex(index=min_samples, columns=min_cluster_size)
null_std_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="permutation_null_std").reindex(index=min_samples, columns=min_cluster_size)
noise_frac_pivot = knn_grid_df.pivot(index="min_samples", columns="min_cluster_size", values="noise_fraction").reindex(index=min_samples, columns=min_cluster_size)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)
plots = [
    (real_pivot, "KNN agreement", "agreement"),
    (delta_pivot, "Agreement minus permutation baseline", "agreement - null mean"),
    (null_std_pivot, "Permutation baseline std", "std"),
    (noise_frac_pivot, "Noise fraction", "fraction of points labeled -1"),
]

for ax, (pivot, title, cbar_label) in zip(axes, plots):
    data = pivot.to_numpy(dtype=float)
    masked = np.ma.masked_invalid(data)
    im = ax.imshow(masked, origin="lower", aspect="auto", cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("min_cluster_size")
    ax.set_ylabel("min_samples")
    
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            value = data[i, j]
            if np.isfinite(value):
                if cbar_label == "fraction of points labeled -1":
                    ax.text(j, i, f"{value:.2f}", ha="center", va="center", color="white", fontsize=9)
                else:
                    ax.text(j, i, f"{value:.3f}", ha="center", va="center", color="white", fontsize=9)
    
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(cbar_label)

fig.savefig(f"lov_eval_cluster_{plm}.png", dpi=600, bbox_inches="tight")
fig.savefig(f"lov_eval_cluster_{plm}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
label_correlations_df = compute_label_correlations(labels_by_params)
print(label_correlations_df)